# 07 — Monte Carlo inference for our selected exponential model

This notebook connects the 100-seed PySR stability experiment to a new Monte Carlo analysis. PySR selects the model family; a deterministic covariance-weighted fit estimates its coefficients. The paper expression is retained only as a reference curve.

## 1. Statistical scope

The primary model family is $f(z;A,B)=A\exp(Bz)$. It is selected because `paper_like_exponential` is the modal class in the targeted `maxsize=7` 100-seed experiment with the public covariance. This evidence is conditional on that PySR configuration; 66/100 is a search frequency, not a posterior model probability.

Two Monte Carlo centers are kept separate:

- `best_fit_expression`: parametric bootstrap centered on our deterministic best-fit curve; this is the primary analysis.
- `observed_data`: perturbations centered directly on the six measurements; this is a sensitivity analysis.

Every realization is refitted for $A$ and $B$. Results are saved both without filtering and after removing only realizations with a grid-bracketed pole. No additional $\epsilon$ threshold is applied. The pole-free result remains a conditional diagnostic rather than a replacement for the unfiltered primary result.

In [ ]:
from pathlib import Path
import json
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize_scalar

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)

## 2. Inputs and experiment directory

In [ ]:
STABILITY_EXPERIMENT_ID = "targeted_maxsize7_100seeds_final_20260906"
EXPERIMENT_ID = "public_full_exponential_20000mc_polefree_20260911"


def find_project_root(start: Path) -> Path:
    """Return the nearest parent containing the required upstream results."""
    for candidate in (start, *start.parents):
        data_file = candidate / "results" / "01_desi_data_check" / "desi_dr2_dh_over_rd.npz"
        stability_file = (
            candidate / "results" / "03_desi_pysr_seed_stability"
            / STABILITY_EXPERIMENT_ID / "classified_summary.csv"
        )
        if data_file.is_file() and stability_file.is_file():
            return candidate
    raise FileNotFoundError("Could not locate Notebook 01 and Notebook 03 results.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_FILE = PROJECT_ROOT / "results" / "01_desi_data_check" / "desi_dr2_dh_over_rd.npz"
STABILITY_FILE = (
    PROJECT_ROOT / "results" / "03_desi_pysr_seed_stability"
    / STABILITY_EXPERIMENT_ID / "classified_summary.csv"
)
NOTEBOOK_RESULTS_DIR = PROJECT_ROOT / "results" / "07_our_exponential_monte_carlo"
RESULTS_DIR = NOTEBOOK_RESULTS_DIR / EXPERIMENT_ID
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
for directory in (NOTEBOOK_RESULTS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment: {EXPERIMENT_ID}")
print(f"Output directory: {RESULTS_DIR}")

## 3. Load the data and model-selection evidence

In [ ]:
with np.load(DATA_FILE) as data:
    z_data = data["z"].astype(float)
    y_data = data["y"].astype(float)
    public_covariance = data["covariance"].astype(float)
    public_sigma = data["sigma"].astype(float)
    tracers = data["tracers"].astype(str)

classified_stability = pd.read_csv(STABILITY_FILE)
public_stability = classified_stability.query("covariance == 'public_full'").copy()
class_counts = (
    public_stability["model_class"].value_counts().rename_axis("model_class")
    .reset_index(name="count")
)
class_counts["fraction"] = class_counts["count"] / len(public_stability)
modal_class = class_counts.iloc[0]["model_class"]
exponential_hits = int((public_stability["model_class"] == "paper_like_exponential").sum())

assert len(public_stability) == 100
assert modal_class == "paper_like_exponential"
assert exponential_hits == 66
display(class_counts)
display(pd.DataFrame({
    "tracer": tracers, "z": z_data, "DH_over_rd": y_data, "sigma_public": public_sigma
}))

## 4. Exponential fit and physical reconstruction

For fixed $B$, the generalized least-squares amplitude has a closed form. The code profiles out $A$ and minimizes

$$\chi^2(A,B)=[\mathbf y-\mathbf f(A,B)]^T C^{-1}[\mathbf y-\mathbf f(A,B)].$$

The reconstruction then applies the same Eq. (3) implementation used in Notebooks 5 and 6. The denominator diagnostic is

$$g(z)=1-\Omega_m\left(\frac{100r_d}{c}\right)^2(1+z)^3f(z)^2.$$

In [ ]:
PAPER_A, PAPER_B = 28.4, -0.519
OMEGA_M = 0.30
RD_MEAN_HINV_MPC = 101.0
RD_SIGMA_HINV_MPC = 2.3
C_KM_S = 299_792.458
Z_GRID = np.linspace(0.0, 2.5, 501)
INVERSE_PUBLIC_COVARIANCE = np.linalg.inv(public_covariance)
QUANTILES = np.array([0.1349898, 2.2750132, 15.8655254, 50.0, 84.1344746, 97.7249868, 99.8650102])
QUANTILE_LABELS = ("q0p135", "q2p275", "q15p866", "median", "q84p134", "q97p725", "q99p865")


def exponential_distance(redshift, amplitude, slope):
    """Evaluate A exp(B z)."""
    return amplitude * np.exp(slope * np.asarray(redshift, dtype=float))


def fit_exponential_profile(observed):
    """Fit A exp(B z) with the full public covariance."""
    observed = np.asarray(observed, dtype=float)

    def profiled_chi2(slope):
        basis = np.exp(slope * z_data)
        amplitude = (basis @ INVERSE_PUBLIC_COVARIANCE @ observed) / (basis @ INVERSE_PUBLIC_COVARIANCE @ basis)
        residual = amplitude * basis - observed
        return float(residual @ INVERSE_PUBLIC_COVARIANCE @ residual)

    result = minimize_scalar(
        profiled_chi2, bounds=(-2.0, 0.0), method="bounded",
        options={"xatol": 1e-11, "maxiter": 200},
    )
    slope = float(result.x)
    basis = np.exp(slope * z_data)
    amplitude = float(
        (basis @ INVERSE_PUBLIC_COVARIANCE @ observed)
        / (basis @ INVERSE_PUBLIC_COVARIANCE @ basis)
    )
    return amplitude, slope, float(result.fun), bool(result.success)


def reconstruct_ensemble(amplitudes, slopes, rd_values, redshift=Z_GRID):
    """Return distance, w(z), and the dimensionless denominator factor."""
    amplitudes = np.asarray(amplitudes, dtype=float)[:, None]
    slopes = np.asarray(slopes, dtype=float)[:, None]
    rd_values = np.asarray(rd_values, dtype=float)[:, None]
    redshift = np.asarray(redshift, dtype=float)[None, :]
    distance = amplitudes * np.exp(slopes * redshift)
    derivative = slopes * distance
    gamma = OMEGA_M * (100.0 * rd_values / C_KM_S) ** 2
    matter_term = gamma * (1.0 + redshift) ** 3 * distance**3
    denominator = 3.0 * (distance - matter_term)
    numerator = 2.0 * (1.0 + redshift) * derivative + 3.0 * matter_term
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        equation_of_state = -1.0 - numerator / denominator
    equation_of_state[~np.isfinite(equation_of_state)] = np.nan
    pole_factor = 1.0 - gamma * (1.0 + redshift) ** 3 * distance**2
    return distance, equation_of_state, pole_factor


def reconstruct_single(amplitude, slope, rd_value=RD_MEAN_HINV_MPC):
    distance, equation_of_state, pole_factor = reconstruct_ensemble([amplitude], [slope], [rd_value])
    return distance[0], equation_of_state[0], pole_factor[0]

## 5. Deterministic coefficient fit

The model class comes from the stability experiment. The coefficients below do not come from selecting one favorable seed; they are re-estimated from the six DESI measurements with the full covariance.

In [ ]:
BEST_A, BEST_B, BEST_CHI2, fit_success = fit_exponential_profile(y_data)
BEST_LOSS = BEST_CHI2 / len(z_data)
best_fit_at_data = exponential_distance(z_data, BEST_A, BEST_B)
paper_at_data = exponential_distance(z_data, PAPER_A, PAPER_B)
paper_residual = paper_at_data - y_data
PAPER_CHI2 = float(paper_residual @ INVERSE_PUBLIC_COVARIANCE @ paper_residual)

deterministic_summary = pd.DataFrame([
    {"model": "our_public_full_fit", "A": BEST_A, "B": BEST_B, "chi2": BEST_CHI2, "loss": BEST_LOSS},
    {"model": "paper_expression", "A": PAPER_A, "B": PAPER_B, "chi2": PAPER_CHI2, "loss": PAPER_CHI2 / len(z_data)},
])
assert fit_success
assert np.isclose(BEST_A, 28.120222, atol=1e-5)
assert np.isclose(BEST_B, -0.508294, atol=1e-5)
display(deterministic_summary)
print(f"Primary model: DH/rd = {BEST_A:.6f} * exp({BEST_B:.6f} z)")

## 6. Monte Carlo configuration

The primary center is the fitted expression. The observed-data center is retained only to show how the centering convention changes the result. Sound-horizon draws use the same independent Gaussian assumption as Notebook 5 so that the analyses remain comparable.

In [ ]:
N_REALIZATIONS = 20_000
RANDOM_SEED = 20260911
CENTER_MODES = ("best_fit_expression", "observed_data")
MOCK_CENTERS = {
    "best_fit_expression": best_fit_at_data,
    "observed_data": y_data,
}
SELECTIONS = ("all", "pole_free")

print(f"Realizations per center: {N_REALIZATIONS:,}")
print(f"Total exponential refits: {N_REALIZATIONS * len(CENTER_MODES):,}")
print("Pole-free diagnostic: remove only realizations with a grid-bracketed pole")

## 7. Run and refit every realization

For each realization, draw six correlated measurements from the public covariance, refit $A$ and $B$, draw $r_d$, reconstruct the curves, diagnose the denominator, and then calculate both unfiltered and pole-free bands. Removed samples are not replenished.

In [ ]:
start_time = time.perf_counter()
seed_sequences = np.random.SeedSequence(RANDOM_SEED).spawn(len(CENTER_MODES))
fit_frames = []
parameter_summary_rows = []
selection_summary_rows = []
band_records = []
band_arrays = {}

for center_index, center_mode in enumerate(CENTER_MODES):
    rng = np.random.default_rng(seed_sequences[center_index])
    mock_data = rng.multivariate_normal(MOCK_CENTERS[center_mode], public_covariance, size=N_REALIZATIONS)
    rd_draws = rng.normal(RD_MEAN_HINV_MPC, RD_SIGMA_HINV_MPC, size=N_REALIZATIONS)
    amplitudes = np.empty(N_REALIZATIONS)
    slopes = np.empty(N_REALIZATIONS)
    chi2_values = np.empty(N_REALIZATIONS)
    successes = np.empty(N_REALIZATIONS, dtype=bool)

    for index, mock_observation in enumerate(mock_data):
        amplitudes[index], slopes[index], chi2_values[index], successes[index] = (
            fit_exponential_profile(mock_observation)
        )

    valid = successes & np.isfinite(amplitudes) & np.isfinite(slopes) & (amplitudes > 0.0) & (rd_draws > 0.0)
    valid_indices = np.flatnonzero(valid)
    valid_a, valid_b, valid_rd = amplitudes[valid], slopes[valid], rd_draws[valid]
    distance, w_values, pole_factor = reconstruct_ensemble(valid_a, valid_b, valid_rd)
    sign_crossings = pole_factor[:, :-1] * pole_factor[:, 1:] <= 0.0
    has_pole = np.any(sign_crossings, axis=1)
    minimum_absolute_g = np.nanmin(np.abs(pole_factor), axis=1)

    fit_frames.append(pd.DataFrame({
        "center_mode": center_mode, "realization": valid_indices,
        "A": valid_a, "B": valid_b, "rd_hinv_mpc": valid_rd,
        "chi2": chi2_values[valid], "pole_in_0_to_2p5": has_pole,
        "minimum_absolute_g": minimum_absolute_g,
    }))
    parameter_summary_rows.append({
        "center_mode": center_mode, "valid_fits": len(valid_a),
        "A_median": np.median(valid_a), "A_std": np.std(valid_a, ddof=1),
        "B_median": np.median(valid_b), "B_std": np.std(valid_b, ddof=1),
        "rd_median": np.median(valid_rd), "rd_std": np.std(valid_rd, ddof=1),
        "pole_fraction_0_to_2p5": np.mean(has_pole),
    })

    selection_masks = {
        "all": np.ones(len(valid_a), dtype=bool),
        "pole_free": ~has_pole,
    }
    for selection_name, selection_mask in selection_masks.items():
        distance_q = np.nanpercentile(distance[selection_mask], QUANTILES, axis=0)
        w_q = np.nanpercentile(w_values[selection_mask], QUANTILES, axis=0)
        band_arrays[(center_mode, selection_name)] = {"distance": distance_q, "w": w_q}
        selection_summary_rows.append({
            "center_mode": center_mode, "selection": selection_name,
            "retained": int(selection_mask.sum()),
            "retained_fraction": float(selection_mask.mean()),
            "maximum_1sigma_w_width": float(np.nanmax(w_q[4] - w_q[2])),
            "maximum_2sigma_w_width": float(np.nanmax(w_q[5] - w_q[1])),
            "maximum_3sigma_w_width": float(np.nanmax(w_q[6] - w_q[0])),
        })
        for grid_index, redshift in enumerate(Z_GRID):
            row = {"center_mode": center_mode, "selection": selection_name, "z": redshift}
            for quantile_index, label in enumerate(QUANTILE_LABELS):
                row[f"distance_{label}"] = distance_q[quantile_index, grid_index]
                row[f"w_{label}"] = w_q[quantile_index, grid_index]
            band_records.append(row)

    print(f"Completed {center_mode}: {valid.sum():,} valid fits; pole fraction {has_pole.mean():.1%}")

elapsed_seconds = time.perf_counter() - start_time
fit_samples = pd.concat(fit_frames, ignore_index=True)
parameter_summary = pd.DataFrame(parameter_summary_rows)
selection_summary = pd.DataFrame(selection_summary_rows)
bands_table = pd.DataFrame(band_records)
print(f"Monte Carlo runtime: {elapsed_seconds:.1f} s")
display(parameter_summary)
display(selection_summary)

## 8. Compare unfiltered and pole-free bands

Each figure keeps the unfiltered result in the first row and the pole-free diagnostic in the second. No minimum-distance threshold is imposed. The displayed $w(z)$ range is clipped for readability; the saved tables retain the actual quantiles.

In [ ]:
our_distance, our_w, _ = reconstruct_single(BEST_A, BEST_B)
paper_distance, paper_w, _ = reconstruct_single(PAPER_A, PAPER_B)
figure_paths = []

for center_mode in CENTER_MODES:
    figure, axes = plt.subplots(2, 2, figsize=(13, 13), sharex=False)
    for row_index, selection_name in enumerate(SELECTIONS):
        distance_q = band_arrays[(center_mode, selection_name)]["distance"]
        w_q = band_arrays[(center_mode, selection_name)]["w"]
        distance_axis, w_axis = axes[row_index]
        for low, high, alpha_value, label in ((0, 6, 0.16, "3 sigma"), (1, 5, 0.26, "2 sigma"), (2, 4, 0.40, "1 sigma")):
            distance_axis.fill_between(Z_GRID, distance_q[low], distance_q[high], color="tab:blue", alpha=alpha_value, label=label)
            w_axis.fill_between(Z_GRID, w_q[low], w_q[high], color="tab:blue", alpha=alpha_value, label=label)
        distance_axis.plot(Z_GRID, distance_q[3], color="tab:blue", lw=2.2, label="MC median")
        distance_axis.plot(Z_GRID, our_distance, color="black", ls="--", lw=2.0, label="Our best-fit expression")
        distance_axis.plot(Z_GRID, paper_distance, color="tab:orange", ls=":", lw=2.0, label="Paper expression")
        distance_axis.errorbar(z_data, y_data, yerr=public_sigma, fmt="o", color="black", capsize=3, label="DESI DR2")
        w_axis.plot(Z_GRID, w_q[3], color="tab:blue", lw=2.2, label="MC median")
        w_axis.plot(Z_GRID, our_w, color="black", ls="--", lw=2.0, label="Our best-fit expression")
        w_axis.plot(Z_GRID, paper_w, color="tab:orange", ls=":", lw=2.0, label="Paper expression")
        w_axis.axhline(-1.0, color="tab:red", ls="--", lw=1.7, label="LCDM")
        row_label = "Unfiltered" if selection_name == "all" else "Pole-free: no grid-bracketed crossing"
        distance_axis.set_ylabel(f"{row_label}\nDH/rd")
        w_axis.set_ylabel(f"{row_label}\nw(z)")
        distance_axis.set_xlabel("Redshift z")
        w_axis.set_xlabel("Redshift z")
        distance_axis.set_xlim(0.0, 2.5)
        w_axis.set_xlim(0.0, 2.5)
        distance_axis.set_xticks(np.arange(0.0, 2.51, 0.5))
        w_axis.set_xticks(np.arange(0.0, 2.51, 0.5))
        w_limits = (-2.0, 0.5) if center_mode == "best_fit_expression" else (-5.0, 3.0)
        w_axis.set_ylim(*w_limits)
        distance_axis.set_box_aspect(1)
        w_axis.set_box_aspect(1)
        distance_axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=7.5)
        w_axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=7.5)
    analysis_label = "Primary parametric bootstrap" if center_mode == "best_fit_expression" else "Observed-center sensitivity analysis"
    figure.suptitle(f"Our exponential model — {analysis_label}", fontsize=15)
    figure.tight_layout(pad=2.0, h_pad=5.5, w_pad=2.0)
    figure_path = FIGURES_DIR / f"{center_mode}_all_vs_pole_free.png"
    figure.savefig(figure_path, dpi=180, bbox_inches="tight")
    figure_paths.append(figure_path)
    plt.show()
    print(f"Saved: {figure_path}")

## 9. Selected-redshift diagnostics

These are pointwise intervals. Inclusion or exclusion of $w=-1$ at individual redshifts is not a global significance test for the complete curve.

In [ ]:
SELECTED_REDSHIFTS = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5])
diagnostic_rows = []
for (center_mode, selection_name), arrays in band_arrays.items():
    for redshift in SELECTED_REDSHIFTS:
        grid_index = int(np.argmin(np.abs(Z_GRID - redshift)))
        q = arrays["w"][:, grid_index]
        diagnostic_rows.append({
            "center_mode": center_mode, "selection": selection_name,
            "z": Z_GRID[grid_index], "w_median": q[3],
            "w_1sigma_low": q[2], "w_1sigma_high": q[4],
            "w_2sigma_low": q[1], "w_2sigma_high": q[5],
            "w_3sigma_low": q[0], "w_3sigma_high": q[6],
            "lcdm_in_1sigma": bool(q[2] <= -1.0 <= q[4]),
            "lcdm_in_2sigma": bool(q[1] <= -1.0 <= q[5]),
            "lcdm_in_3sigma": bool(q[0] <= -1.0 <= q[6]),
        })
diagnostic_table = pd.DataFrame(diagnostic_rows)
display(diagnostic_table)

## 10. Save reproducible outputs

In [ ]:
FIT_SAMPLES_FILE = TABLES_DIR / "monte_carlo_fit_samples.csv.gz"
PARAMETER_SUMMARY_FILE = TABLES_DIR / "parameter_summary.csv"
SELECTION_SUMMARY_FILE = TABLES_DIR / "selection_summary.csv"
BANDS_FILE = TABLES_DIR / "monte_carlo_bands.csv"
DIAGNOSTIC_FILE = TABLES_DIR / "selected_redshift_diagnostics.csv"
DETERMINISTIC_FILE = TABLES_DIR / "deterministic_models.csv"
MODEL_SELECTION_FILE = TABLES_DIR / "model_selection_evidence.csv"
ARRAY_FILE = RESULTS_DIR / "monte_carlo_bands.npz"
CONFIGURATION_FILE = RESULTS_DIR / "configuration.json"

fit_samples.to_csv(FIT_SAMPLES_FILE, index=False, compression="gzip")
parameter_summary.to_csv(PARAMETER_SUMMARY_FILE, index=False)
selection_summary.to_csv(SELECTION_SUMMARY_FILE, index=False)
bands_table.to_csv(BANDS_FILE, index=False)
diagnostic_table.to_csv(DIAGNOSTIC_FILE, index=False)
deterministic_summary.to_csv(DETERMINISTIC_FILE, index=False)
class_counts.to_csv(MODEL_SELECTION_FILE, index=False)

array_payload = {"z": Z_GRID, "quantiles_percent": QUANTILES}
for (center_mode, selection_name), arrays in band_arrays.items():
    prefix = f"{center_mode}__{selection_name}"
    array_payload[f"{prefix}__distance"] = arrays["distance"]
    array_payload[f"{prefix}__w"] = arrays["w"]
np.savez_compressed(ARRAY_FILE, **array_payload)

configuration = {
    "experiment_id": EXPERIMENT_ID,
    "source_stability_experiment": STABILITY_EXPERIMENT_ID,
    "covariance": "public_full",
    "model_selection_rule": "modal model class across the targeted maxsize=7 100-seed public-covariance experiment",
    "model_selection_result": {"class": modal_class, "hits": exponential_hits, "runs": len(public_stability)},
    "model_family": "A * exp(B z)",
    "coefficient_estimation": "generalized least squares after fixing the model family",
    "best_fit_parameters": {"A": BEST_A, "B": BEST_B, "chi2": BEST_CHI2},
    "paper_reference": {"A": PAPER_A, "B": PAPER_B},
    "n_realizations_per_center": N_REALIZATIONS,
    "random_seed": RANDOM_SEED,
    "center_modes": list(CENTER_MODES),
    "primary_center": "best_fit_expression",
    "slope_search_bounds": [-2.0, 0.0],
    "selections": {"all": "unfiltered", "pole_free": "remove only grid-bracketed pole crossings"},
    "cosmology": {
        "omega_m_fixed": OMEGA_M, "rd_distribution": "independent Gaussian",
        "rd_mean_hinv_mpc": RD_MEAN_HINV_MPC, "rd_sigma_hinv_mpc": RD_SIGMA_HINV_MPC,
        "c_km_s": C_KM_S,
    },
    "pointwise_quantiles_percent": QUANTILES.tolist(),
    "runtime_seconds": elapsed_seconds,
    "interpretation": (
        "Monte Carlo uncertainty conditional on the exponential model family. "
        "PySR structural uncertainty is not included; pole-free results are conditional diagnostics."
    ),
    "software_versions": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
    },
}
with CONFIGURATION_FILE.open("w", encoding="utf-8") as stream:
    json.dump(configuration, stream, indent=2)

for path in (FIT_SAMPLES_FILE, PARAMETER_SUMMARY_FILE, SELECTION_SUMMARY_FILE,
             BANDS_FILE, DIAGNOSTIC_FILE, DETERMINISTIC_FILE, MODEL_SELECTION_FILE,
             ARRAY_FILE, CONFIGURATION_FILE, *figure_paths):
    print(f"Saved: {path}")

## 11. Evaluation rules

- Treat the unfiltered `best_fit_expression` result as the primary parametric-bootstrap output.
- Treat `observed_data` as a centering sensitivity check, not a second primary answer.
- Report the retained fraction whenever the pole-free result is shown.
- Do not interpret a 66/100 PySR hit rate as a posterior probability.
- Do not claim that these bands include symbolic-model uncertainty.
- Do not tune the model or filter to make the result resemble the paper.
- Pointwise exclusion of $w=-1$ is not evidence for global exclusion of $\Lambda$CDM.